# Duck voice on the A30

Use this notebook to download and audition Qwen3-TTS before training a custom Scottish voice. The test phrases below are synthetic examples. This notebook runs separately from the existing Duck intent router.

## 1. Prepare a CUDA kernel

Use a Python kernel with CUDA-enabled PyTorch. In that kernel install Qwen’s runtime with `%pip install qwen-tts soundfile huggingface_hub`, then restart the kernel. Qwen recommends an isolated environment. The model is downloaded from Hugging Face to this host and generation runs locally. [Setup and package documentation](https://github.com/QwenLM/Qwen3-TTS#environment-setup).

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import HfApi, snapshot_download
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

assert torch.cuda.is_available(), "Select an A30 CUDA-enabled Python kernel."
print(torch.cuda.get_device_name(0))
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
output = Path("duck-voice-audition")
output.mkdir(exist_ok=True)

## 2. Fetch the voice checkpoint and keep its revision

Use the [Qwen3-TTS 0.6B Base checkpoint](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-Base). Hugging Face reports 914,643,008 parameters for the main checkpoint. A separately bundled speech tokenizer is also required. The combined stack exceeds one billion parameters if both components are counted together.

In [ ]:
repo_id = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
manifest_path = output / "revisions.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
hub = HfApi()
info = hub.model_info(repo_id, revision=manifest.get(repo_id))
if info.safetensors is None or not 0 < info.safetensors.total <= 1_000_000_000:
    raise RuntimeError("The main checkpoint must publish a parameter count within the cap.")
snapshot = snapshot_download(repo_id=repo_id, revision=info.sha)
manifest[repo_id] = info.sha
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(repo_id, info.sha, f"{info.safetensors.total:,} main-checkpoint parameters")

## 3. Audition the same farm report in four languages

For each voice you want to audition, add a permitted reference recording and exact transcript under `voice-references/`. Use matching names such as `en-GB-scotland.wav` and `en-GB-scotland.txt`. The other locale names are `es-MX`, `zh-CN`, and `ja-JP`. A native speaker recording for each target locale gives us a reference for accent and clarity. The cell skips locales whose reference files are missing.

In [ ]:
cases = [
    ("en-GB-scotland", "English",
     "On the fourteenth of November, expect minus two degrees Celsius near Auchterarder. Rainfall is twenty-six millimetres, and the field covers twelve point five hectares."),
    ("es-MX", "Spanish",
     "El catorce de noviembre se esperan menos dos grados Celsius cerca de Auchterarder. La precipitación es de veintiséis milímetros y la parcela tiene doce coma cinco hectáreas."),
    ("zh-CN", "Chinese",
     "十一月十四日，奥赫特拉德附近预计为零下二摄氏度。降雨量为二十六毫米，地块面积为十二点五公顷。"),
    ("ja-JP", "Japanese",
     "十一月十四日、オークテラーダー付近では氷点下二度の見込みです。降水量は二十六ミリメートルで、農地の面積は十二点五ヘクタールです。"),
]

references = Path("voice-references")
available = [case for case in cases if all((references / f"{case[0]}.{ext}").is_file() for ext in ("wav", "txt"))]
if not available:
    raise FileNotFoundError("Add at least one permitted reference WAV and exact transcript.")
for locale, *_ in cases:
    if not any(case[0] == locale for case in available):
        print(f"Skipping {locale}: reference recording or transcript missing.")
model = Qwen3TTSModel.from_pretrained(snapshot, device_map="cuda:0", dtype=dtype)
torch.manual_seed(42)
results = []
for locale, language, text in available:
    ref_audio = references / f"{locale}.wav"
    ref_text = (references / f"{locale}.txt").read_text(encoding="utf-8").strip()
    if not ref_text:
        raise ValueError(f"Reference transcript for {locale} is empty.")
    started = time.perf_counter()
    with torch.inference_mode():
        audio, sample_rate = model.generate_voice_clone(text=text, language=language, ref_audio=str(ref_audio), ref_text=ref_text)
    waveform = np.asarray(audio[0])
    if not waveform.size or not np.isfinite(waveform).all() or not np.any(waveform):
        raise RuntimeError(f"Invalid or silent audio for {locale}")
    path = output / f"{locale}.wav"
    sf.write(path, waveform, sample_rate)
    results.append({
        "locale": locale, "file": path.name, "reference": text,
        "seconds_to_generate": round(time.perf_counter() - started, 2),
        "seconds_of_audio": round(len(waveform) / sample_rate, 2),
        "review": None,
    })
    print(locale, text)
    display(Audio(filename=str(path)))

## 4. Review audio and decide what to train

For every language, ask a fluent reviewer to listen on a phone speaker and write down the date, temperature with sign, precipitation and unit, area and unit, and place name. Compare with **14 November, −2 °C, 26 mm, 12.5 ha, Auchterarder**. Record omissions, additions, pronunciation issues and accent feedback. Use these fixtures for an initial comparison. Add unseen examples covering decimals, coordinate signs, farm names, safety messages and missing data for a release evaluation.

In [ ]:
report = output / "audition.json"
report.write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(report.resolve())
for row in results:
    print(row["locale"], row["seconds_to_generate"], "seconds to generate", row["seconds_of_audio"], "seconds of audio")

## 5. Fine-tune and serve

Once the Scottish voice direction is approved, collect recordings and exact transcripts from a speaker who has agreed to model training and deployment. Set the initial checkpoint to Qwen3-TTS 0.6B Base in Qwen’s [single-speaker fine-tuning recipe](https://github.com/QwenLM/Qwen3-TTS/tree/main/finetuning). Rerun the audition against the new checkpoint. Keep a separate check for multilingual voices. Check the parameter count and use the same review method before adding any other language model.

## Serve voice separately from chat

Upload `duck_voice_service.py` and put the approved Scottish reference files in `voice-references/en-GB-scotland.wav` and `voice-references/en-GB-scotland.txt` beside it. Start `python duck_voice_service.py --port 19231` in the CUDA environment. The worker listens on `127.0.0.1:19231` and accepts `POST /speak`. The chatbot continues to use its own service on port 19230.

In your dashboard's `.env.local`, set `DUCK_VOICE_API_URL` to the private address or tunnel URL for the voice worker's `/speak` path. For shared gateways, configure the same `DUCK_VOICE_API_TOKEN` on the worker and dashboard server. Use the `Read aloud` control on a chat answer to test the connection. The dashboard currently requests Scottish English. Localized chat replies need their own translation workflow.

Feedback from the dashboard goes to `.duck-model/feedback.jsonl` on the dashboard server. Review corrections before using them as training examples.
